## Setting up variables

In [14]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
!wget -q https://raw.githubusercontent.com/yarichard/mountain-race/feature/llm/data/evaluate.py -O evaluate.py

In [2]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

In [3]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
PROJECT_NAME = "gear_training"
HF_USER = "yrichard"
DATA_USER = "yrichard"
DATASET_NAME = f"{DATA_USER}/gear_raw"

MAX_SEQUENCE_LENGTH = 768
RUN_NAME = "2026-04-28_13.15.01"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

### Log in to HuggingFace & Load dataset

In [4]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [5]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['validation']
test = dataset['test']

README.md:   0%|          | 0.00/727 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/70.8k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/17.6k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/14.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/50 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/28 [00:00<?, ? examples/s]

## Testing unmerged model ( run these cells)

In [6]:
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )

In [7]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Memory footprint: 2197.6 MB


In [8]:
from peft import PeftModel
import torch

fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)

# Explicitly cast the lm_head to float32 to resolve type mismatch.
# The error "expected scalar type Float but found BFloat16" indicates lm_head expects float32
# but the input it receives (or its own weights) are in bfloat16. Casting to float32 ensures
# consistency for this critical layer.
if hasattr(fine_tuned_model.base_model, 'lm_head') and fine_tuned_model.base_model.lm_head.weight.dtype != torch.float32:
    print(f"Casting lm_head from {fine_tuned_model.base_model.lm_head.weight.dtype} to torch.float32...")
    fine_tuned_model.base_model.lm_head = fine_tuned_model.base_model.lm_head.to(torch.float32)

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/48.7M [00:00<?, ?B/s]

Casting lm_head from torch.bfloat16 to torch.float32...


## Testing merged model (run these cells)

In [ ]:
MERGED_PROJECT_RUN_NAME = f"{PROJECT_RUN_NAME}-merged"
HUB_MODEL_NAME = f"{HF_USER}/{MERGED_PROJECT_RUN_NAME}"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

fine_tuned_model = AutoModelForCausalLM.from_pretrained(
    HUB_MODEL_NAME,
    device_map="auto",
)
fine_tuned_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory footprint: 6425.5 MB


## Infer model

In [9]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=MAX_SEQUENCE_LENGTH)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [10]:
gear = test[7]
print(gear["prompt"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 28 Apr 2026

You are a mountain climbing equipment assistant. Parse the following gear description and return a JSON array. Each element must have exactly three fields:
	- "name": equipment name (string, in french)
	- "quantity": number needed (integer, 1 if unspecified)
	- "notes": "optional" or "mandatory" (translated in french), plus any relevant detail (string, in french)
The name of these equipments are related with the mountain activities. You should only point out personal equipment, for instance quickdraws or rope.
You should include only equipment you're absolutely sure about. Output ONLY the JSON array, no explanation.<|eot_id|><|start_header_id|>user<|end_header_id|>

Gear description:
 2 jeux complets Friends jusqu’au 4.
1 numéro 5 (facultatif).
1 jeu de câblés.
Corde 50 m 
1 Marteau et 4-5 pitons (au cas où)
Gants de fissure (facultatif)<|eot_id|><|start_header_id

In [11]:
print("Answer:")
model_predict(gear)

Answer:


'[{"name": "Corde", "quantity": 1, "notes": "obligatoire, 50 m"}, {"name": "Friend / coinceur #5", "quantity": 1, "notes": "facultatif"}, {"name": "Câblés / stoppeurs", "quantity": 1, "notes": "facultatif"}, {"name": "Gants", "quantity": 1, "notes": "facultatif"}, {"name": "Marteau", "quantity": 1, "notes": "facultatif"}, {"name": "Pitons", "quantity": 5, "notes": "facultatif"}]<|eot_id|>'

## Evaluate model quality

In [15]:
from evaluate import evaluate
set_seed(42)
evaluate(model_predict, test)